In [2]:
# !pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers transformers

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os 

C:\Users\User\AppData\Local\Temp\ipykernel_28936\3003692989.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
# 1. Load the text files from your folder
folder_path = "my_txt_files" 
documents = []

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        # Using utf-8 encoding is crucial for reading Arabic characters properly
        loader = TextLoader(file_path, encoding="utf-8")
        documents.extend(loader.load())

# 2. Split documents into manageable chunks
# Since AraBERT has a 512-token limit, keeping chunk sizes around 300-400 characters works well
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# 3. Initialize AraBERT Embeddings
# Using aubmindlab's AraBERT v0.2 model wrapped in HuggingFaceEmbeddings
model_name = "aubmindlab/bert-base-arabertv02"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'} # Change to 'cuda' if you have a GPU available
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'my_txt_files'

In [ ]:
# 4. Initialize Chroma Database (matching your exact configuration)
db = Chroma(
    embedding_function=embeddings,
    collection_name="my_rag_project",
    persist_directory="./my_vector_db",
    collection_metadata={"hnsw:space": "cosine"} # CRUCIAL FOR ARABIC SEMANTICS
)

# 5. Add documents in small, bite-sized batches of 10 chunks
batch_size = 10
for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    db.add_documents(batch)
    print(f"Successfully processed chunks {i} to {i + len(batch)}")

print("==== Ingestion Complete! Data saved permanently to disk ====")

In [ ]:
!pip install --upgrade google-generativeai

In [ ]:
import os
import google.generativeai as genai
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Configure the Gemini API with your Free Key
API_KEY = "API"
genai.configure(api_key=API_KEY)

# 2. Initialize your exact AraBERT Embedding Function
print("Loading AraBERT Embedding model...")
model_name = "aubmindlab/bert-base-arabertv02"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'} 
)

# 3. Connect to your existing saved Chroma Database
print("Connecting to Chroma Vector Store...")
db = Chroma(
    embedding_function=embeddings,
    collection_name="my_rag_project",
    persist_directory="./my_vector_db"
)


def ask_chroma_books_stream(query):
    try:
        # Retrieve the top 4 closest matching text chunks
        docs = db.similarity_search(query, k=4)
        
        if not docs:
            print("\nالإجابة: المعلومة غير متوفرة في الكتب المرفقة.")
            return
            
        context_text = "\n\n".join([doc.page_content for doc in docs])
        
        try:
            file_source = os.path.basename(docs[0].metadata.get('source', 'الكتب المخزنة'))
        except Exception:
            file_source = "الكتب المخزنة"

        # Balanced prompt allowing natural phrasing reasoning
        system_instruction = (
            "أنت مساعد ذكي ومتخصص في تحليل النصوص العربية ومحدد جداً. مهمتك هي الإجابة على سؤال المستخدم بناءً على السياق المستخرج المقدم فقط.\n"
            "شروط الإجابة:\n"
            "1. يجب أن تعتمد إجابتك بالكامل على السياق المرفق أدناه.\n"
            "2. لا تضف أي معلومات خارجية تماماً من خارج هذا النص.\n"
            "3. يمكنك فهم المعنى المرادف لغوياً بدقة.\n"
            "4. إذا لم تكن الإجابة واضحة بشكل مباشر، يجب عليك تحليل نص الكتب بدقة واستنباط الإجابة الصحيحة بناءً على الفهم والربط المنطقي بين الأفكار الواردة.\n"
            "5. يجب عليك دائماً ذكر اسم الكتاب أو المصدر الذي استخرجت منه الإجابة في نهاية ردك."
        )

        
        user_prompt = f"""
{system_instruction}

[السياق المستخرج من - المصدر: {file_source}]:
{context_text}

[سؤال المستخدم]:
{query}

الإجابة المباشرة:
"""

        # 4. Call the model and set stream=True
        model = genai.GenerativeModel('gemini-3.6-flash')
        response = model.generate_content(user_prompt, stream=True)
        
        print("\nالإجابة: ", end="", flush=True)
        
        # Loop through chunks and print them immediately to your notebook output
        for chunk in response:
            print(chunk.text, end="", flush=True)
        print("\n") # New line after generation completes

    except Exception as e:
        print(f"\nحدث خطأ أثناء المعالجة: {str(e)}")


# 5. Interactive loop to chat with your books
if __name__ == "__main__":
    print("\n==== محرك البث الفوري جاهز تماماً! ====")
    print("يمكنك البدء بطرح الأسئلة (اكتب 'خروج' للإنهاء):")
    
    while True:
        user_query = input("\nسؤالك باللغة العربية: ")
        if user_query.strip() in ["خروج", "exit", "quit"]:
            print("تم إغلاق البرنامج بنجاح.")
            break
            
        if not user_query.strip():
            continue
            
        print("جاري البحث دلالياً...")
        ask_chroma_books_stream(user_query)
